# Testing for using FastF1 library for historical F1 data retrieval.

Imports & disable logging/warnings:

In [1]:
import fastf1 as ff1
import numpy as np
import pandas as pd
import os
import warnings
import logging
import time

logging.getLogger("fastf1").setLevel(logging.DEBUG)
warnings.filterwarnings("ignore", category=FutureWarning)

Get all qualifying data from 2018-2025:

In [ ]:
cache_dir = "../../data/raw/historical/"
os.makedirs(cache_dir, exist_ok=True)
ff1.Cache.enable_cache(cache_dir)

YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
SESSION = "Q"

all_data = []
for year in YEARS:
    schedule = None
    try:
        schedule = ff1.get_event_schedule(year, include_testing=False)
    except Exception as e:
        print(f"Could not load schedule for {year}. Reason: {e}")
        continue

    for _, event in schedule.iterrows():
        try:
            quali = ff1.get_session(year, event.EventName, SESSION)
            quali.load(laps=True)
        except Exception as e:
            print(f"Skipping {year} {event.EventName}. Reason: {e}")
            continue
        
        if quali.laps.empty:
            print(f"No qualifying laps for {year} {event.EventName}. Skipping.")
            continue

        all_data.append(quali.laps)
        print(f"Loaded {year} {event.EventName}.")
        time.sleep(1) 